# Expressions

Split from `01_operators_and_expressions.ipynb` to keep this topic self-contained.

**Navigation:** [Previous: Operators](./01_operators.ipynb) · [Topic overview](./01_operators_and_expressions.ipynb)


<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
These patterns are used to build reusable data-processing scripts for FASTA/VCF/TSV handling, QC summaries, and automation glue code.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Mutable vs immutable objects and how that affects function behavior.
- Vectorized/dataframe workflows vs loops: both are useful, but for different bottlenecks.
- Reading errors: most failures come from dtype mismatches and unexpected missing values.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


# Module 3: Operators and Expressions

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Use arithmetic operators for biological calculations
2. Compare values with comparison operators
3. Combine conditions with logical operators
4. Test membership with `in` and `not in`
5. Manipulate sequences with string operators (concatenation, repetition, slicing)
6. Understand operator precedence

---

---

[← Previous: Module 2: Variables and Data Types](../../Tier_1_Python_for_Bioinformatics/02_Variables_and_Data_Types/01_variables_and_data_types.ipynb) | [Next: Module 4: Strings and Biological Sequences →](../../Tier_1_Python_for_Bioinformatics/04_Strings_and_Sequences/01_strings_and_sequences.ipynb)

---

## 1. Arithmetic Operators

| Operator | Name | Example | Result |
|----------|------|---------|--------|
| `+` | Addition | `5 + 3` | `8` |
| `-` | Subtraction | `5 - 3` | `2` |
| `*` | Multiplication | `5 * 3` | `15` |
| `/` | Division | `5 / 2` | `2.5` |
| `//` | Floor division | `5 // 2` | `2` |
| `%` | Modulo (remainder) | `5 % 2` | `1` |
| `**` | Exponentiation | `5 ** 2` | `25` |

In [ ]:
# Basic arithmetic
a = 15
b = 4

print(f"{a} + {b}  = {a + b}")
print(f"{a} - {b}  = {a - b}")
print(f"{a} * {b}  = {a * b}")
print(f"{a} / {b}  = {a / b}")      # always returns float
print(f"{a} // {b} = {a // b}")     # truncates toward negative infinity
print(f"{a} % {b}  = {a % b}")      # remainder
print(f"{a} ** 2 = {a ** 2}")       # exponentiation

### Division: `/` vs `//`

This distinction matters constantly in bioinformatics:
- `/` (true division) always returns a `float`
- `//` (floor division) returns an `int` when both operands are `int`

In [ ]:
# True division vs floor division
print(f"7 / 2  = {7 / 2}")     # 3.5  (float)
print(f"7 // 2 = {7 // 2}")    # 3    (int)
print(f"7 % 2  = {7 % 2}")     # 1    (remainder)

# Verify: (a // b) * b + (a % b) == a
a, b = 7, 2
print(f"\nVerification: {a // b} * {b} + {a % b} = {(a // b) * b + (a % b)} == {a}")

### Bio application: Codons and reading frames

The modulo operator `%` and floor division `//` are essential for working with codons (triplets of nucleotides).

In [ ]:
# How many complete codons in a sequence?
seq_length = 1000

complete_codons = seq_length // 3
remaining_nucleotides = seq_length % 3

print(f"Sequence length: {seq_length} nt")
print(f"Complete codons: {complete_codons}")
print(f"Leftover nucleotides: {remaining_nucleotides}")

In [ ]:
# Which reading frame is a given position in?
# Reading frames: 0, 1, 2
positions = [0, 1, 2, 3, 6, 10, 47]

print(f"{'Position':>10} {'Reading Frame':>15}")
print("-" * 27)
for pos in positions:
    frame = pos % 3
    print(f"{pos:>10} {frame:>15}")

### Bio application: GC content calculation

GC content = (G + C) / total length * 100%

In [ ]:
sequence = "ATGCGATCGATCGTAGC"

g_count = sequence.count("G")
c_count = sequence.count("C")
total = len(sequence)

gc_content = (g_count + c_count) / total * 100

print(f"Sequence: {sequence}")
print(f"G: {g_count}, C: {c_count}, Total: {total}")
print(f"GC content: {gc_content:.2f}%")

### Bio application: Protein molecular weight

A rough estimate of protein molecular weight:
- Average amino acid mass: ~110 Da
- Water lost per peptide bond: 18 Da
- Number of peptide bonds = n - 1 (for n amino acids)

In [ ]:
num_amino_acids = 393   # human p53 protein
avg_aa_mass = 110       # Daltons
water_mass = 18         # Daltons

protein_mw = (num_amino_acids * avg_aa_mass) - ((num_amino_acids - 1) * water_mass)

print(f"Protein: p53 ({num_amino_acids} amino acids)")
print(f"Estimated MW: {protein_mw:,} Da")
print(f"Estimated MW: {protein_mw / 1000:.1f} kDa")
print(f"(Actual MW of p53: ~43.7 kDa)")

### Augmented assignment operators

Python provides shorthand for updating a variable:

| Shorthand | Equivalent |
|-----------|------------|
| `x += 5` | `x = x + 5` |
| `x -= 3` | `x = x - 3` |
| `x *= 2` | `x = x * 2` |
| `x /= 4` | `x = x / 4` |
| `x //= 3` | `x = x // 3` |
| `x %= 3` | `x = x % 3` |
| `x **= 2` | `x = x ** 2` |

In [ ]:
# Count GC nucleotides using augmented assignment
sequence = "ATGCGATCGATCGTAGC"
gc_count = 0

for nucleotide in sequence:
    if nucleotide in "GC":
        gc_count += 1    # same as gc_count = gc_count + 1

print(f"GC count: {gc_count}")
print(f"GC content: {gc_count / len(sequence) * 100:.1f}%")

---

## 2. Comparison Operators

Comparison operators return `True` or `False`. They are the foundation of all decision-making in programs.

| Operator | Meaning | Example |
|----------|---------|----------|
| `==` | Equal to | `5 == 5` -> `True` |
| `!=` | Not equal to | `5 != 3` -> `True` |
| `<` | Less than | `3 < 5` -> `True` |
| `>` | Greater than | `5 > 3` -> `True` |
| `<=` | Less than or equal | `5 <= 5` -> `True` |
| `>=` | Greater than or equal | `5 >= 3` -> `True` |

In [ ]:
# Comparing biological values
gc_content = 0.65

print(f"GC content = {gc_content}")
print(f"GC > 0.5?    {gc_content > 0.5}")      # True
print(f"GC < 0.4?    {gc_content < 0.4}")      # False
print(f"GC == 0.65?  {gc_content == 0.65}")    # True
print(f"GC != 0.5?   {gc_content != 0.5}")     # True

### Chained comparisons

Python allows chaining comparisons, which is more readable than using `and`.

In [ ]:
gc_content = 0.52

# Check if GC content is in the "normal" range (40-60%)
# Without chaining:
in_range_1 = gc_content >= 0.40 and gc_content <= 0.60

# With chaining (more Pythonic):
in_range_2 = 0.40 <= gc_content <= 0.60

print(f"GC = {gc_content}")
print(f"In normal range (40-60%)? {in_range_2}")

In [ ]:
# Bio application: classifying GC content
def classify_gc(gc_percent):
    """Classify GC content into biological categories."""
    if gc_percent < 30:
        return "Very AT-rich (e.g., Plasmodium)"
    elif 30 <= gc_percent < 40:
        return "AT-rich"
    elif 40 <= gc_percent < 60:
        return "Moderate (typical for many organisms)"
    elif 60 <= gc_percent < 70:
        return "GC-rich"
    else:
        return "Very GC-rich (e.g., Streptomyces)"

test_values = [19.4, 35.0, 41.0, 50.7, 65.0, 72.0]
for gc in test_values:
    print(f"  GC {gc:.1f}% -> {classify_gc(gc)}")

### Comparing strings

Strings are compared lexicographically (dictionary order), character by character.

In [ ]:
# String comparisons
print(f"'ATG' == 'ATG': {'ATG' == 'ATG'}")   # True (exact match)
print(f"'ATG' == 'atg': {'ATG' == 'atg'}")   # False (case-sensitive!)
print(f"'ATG' < 'GGG':  {'ATG' < 'GGG'}")    # True (A < G in ASCII)

# Case-insensitive comparison
seq1 = "ATGC"
seq2 = "atgc"
print(f"\nCase-insensitive match: {seq1.upper() == seq2.upper()}")

---

## 3. Logical Operators

Logical operators combine boolean expressions:

| Operator | Meaning | Example |
|----------|---------|--------|
| `and` | Both must be True | `True and False` -> `False` |
| `or` | At least one True | `True or False` -> `True` |
| `not` | Inverts the value | `not True` -> `False` |

### Truth tables

```
A     B     A and B    A or B    not A
True  True  True       True      False
True  False False      True      False
False True  False      True      True
False False False      False     True
```

In [ ]:
# Sequence quality filters
seq_length = 1500
gc_content = 0.48
has_ambiguous_bases = False

# A sequence passes QC if:
# - length is between 500 and 3000 bp
# - GC content is between 30% and 70%
# - no ambiguous bases

valid_length = 500 <= seq_length <= 3000
valid_gc = 0.30 <= gc_content <= 0.70
no_ambiguity = not has_ambiguous_bases

passes_qc = valid_length and valid_gc and no_ambiguity

print(f"Length ({seq_length}): valid? {valid_length}")
print(f"GC ({gc_content}):   valid? {valid_gc}")
print(f"No ambiguity:   {no_ambiguity}")
print(f"Passes QC:      {passes_qc}")

In [ ]:
# Bio application: checking if a codon is a stop codon
codon = "TAG"

is_stop = (codon == "TAA") or (codon == "TAG") or (codon == "TGA")
print(f"Is '{codon}' a stop codon? {is_stop}")

# More Pythonic way (using 'in'):
is_stop = codon in ["TAA", "TAG", "TGA"]
print(f"Is '{codon}' a stop codon? {is_stop}")

In [ ]:
# Bio application: filter sequences by multiple criteria
sequences = [
    ("seq_1", "ATGCGATCGATCGATCG", 0.53),
    ("seq_2", "AAATTTAAATTTAAATTT", 0.0),
    ("seq_3", "GCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGC", 1.0),
    ("seq_4", "ATGC", 0.5),
    ("seq_5", "ATGAAACCCGGGTAA", 0.53),
]

print("Sequences passing filters (length >= 10, 0.3 <= GC <= 0.7):")
for name, seq, gc in sequences:
    if len(seq) >= 10 and 0.3 <= gc <= 0.7:
        print(f"  {name}: length={len(seq)}, GC={gc:.0%}")

---

## 4. Membership Operators: `in` and `not in`

The `in` operator checks whether a value exists in a collection (string, list, set, dictionary keys).

In [ ]:
# Check if a substring exists in a DNA sequence
dna = "ATGAAACCCGGGTAA"

print(f"'ATG' in sequence? {'ATG' in dna}")     # True (start codon present)
print(f"'GGG' in sequence? {'GGG' in dna}")     # True
print(f"'TTT' in sequence? {'TTT' in dna}")     # False
print(f"'TTT' not in sequence? {'TTT' not in dna}")  # True

In [ ]:
# Check membership in a list of stop codons
stop_codons = ["TAA", "TAG", "TGA"]

test_codons = ["ATG", "TAA", "GGG", "TGA", "CCC", "TAG"]

for codon in test_codons:
    status = "STOP" if codon in stop_codons else "coding"
    print(f"  {codon} -> {status}")

In [ ]:
# Validate a DNA sequence using 'in'
def is_valid_dna(sequence):
    """Check if a sequence contains only valid DNA nucleotides."""
    valid = "ATGC"
    for char in sequence.upper():
        if char not in valid:
            return False
    return True

print(is_valid_dna("ATGCGATCG"))    # True
print(is_valid_dna("ATGCXYZ"))      # False
print(is_valid_dna("augcgaucg"))    # True (after upper())

In [ ]:
# Membership in dictionaries checks KEYS, not values
codon_table = {
    "ATG": "Met",
    "TAA": "Stop",
    "TAG": "Stop",
    "TGA": "Stop",
    "GGG": "Gly",
}

print(f"'ATG' in codon_table? {'ATG' in codon_table}")     # True (it is a key)
print(f"'Met' in codon_table? {'Met' in codon_table}")     # False (it is a value, not a key)

### Performance note: `in` with sets vs lists

Checking membership in a `set` is much faster than in a `list`. For large collections, always convert to a set first.

In [ ]:
# For small collections, the difference is negligible
# For large collections, sets are dramatically faster

valid_nucleotides_list = ["A", "T", "G", "C"]
valid_nucleotides_set = {"A", "T", "G", "C"}

# Both work the same way:
print("G" in valid_nucleotides_list)  # True (scans the list)
print("G" in valid_nucleotides_set)   # True (hash lookup -- much faster)

# Rule of thumb: use sets when you only need membership testing

---

## 5. String Operators for Sequences

Strings support several operators that are very useful for working with biological sequences.

### Concatenation (`+`)

Join two strings end-to-end.

In [ ]:
# Concatenating sequence fragments
exon1 = "ATGAAA"
exon2 = "CCCGGG"
exon3 = "TTTTAA"

# After splicing, exons are joined
mrna = exon1 + exon2 + exon3
print(f"Exon 1: {exon1}")
print(f"Exon 2: {exon2}")
print(f"Exon 3: {exon3}")
print(f"Spliced mRNA: {mrna}")

In [ ]:
# Building a FASTA entry
header = ">gene_001 Hypothetical protein"
sequence = "ATGCGATCGATCGATCGATCG"

fasta_entry = header + "\n" + sequence
print(fasta_entry)

### Repetition (`*`)

Repeat a string a given number of times.

In [ ]:
# Microsatellite / tandem repeat
repeat_unit = "CAG"
normal_allele = repeat_unit * 20         # normal: ~20 repeats
disease_allele = repeat_unit * 40        # Huntington's: >36 repeats

print(f"Normal allele ({len(normal_allele)} nt):")
print(f"  {normal_allele}")
print(f"\nDisease allele ({len(disease_allele)} nt):")
print(f"  {disease_allele}")

# Polylinker / multiple cloning site separator
separator = "-" * 50
print(f"\n{separator}")

### Slicing (recap with biological context)

Slicing is arguably the single most useful operation for working with biological sequences.

In [ ]:
# Extract all codons from a coding sequence
cds = "ATGAAAGCCCCCGGGTTTCCCAAATAA"

print(f"CDS: {cds}")
print(f"Length: {len(cds)} nt ({len(cds) // 3} codons)")
print()

# Extract each codon
print("Codons:")
for i in range(0, len(cds), 3):
    codon = cds[i:i+3]
    position = i // 3 + 1
    is_stop = codon in ["TAA", "TAG", "TGA"]
    marker = " <-- STOP" if is_stop else ""
    marker = " <-- START" if codon == "ATG" else marker
    print(f"  Codon {position}: {codon}{marker}")

In [ ]:
# Restriction enzyme site search using slicing
# EcoRI recognizes GAATTC

dna = "ATGCGAATTCGATCGATCGATCGAATTCATCG"
enzyme_site = "GAATTC"

print(f"Sequence: {dna}")
print(f"Searching for {enzyme_site} (EcoRI) sites...")

positions = []
for i in range(len(dna) - len(enzyme_site) + 1):
    if dna[i:i + len(enzyme_site)] == enzyme_site:
        positions.append(i)

print(f"Found {len(positions)} site(s) at position(s): {positions}")

---

## 6. Identity Operators: `is` and `is not`

- `==` checks if values are **equal**
- `is` checks if two variables point to the **same object** in memory

In practice, only use `is` to check for `None`.

In [ ]:
# Use 'is' for None checks
gene_annotation = None

if gene_annotation is None:
    print("Gene has no annotation")

if gene_annotation is not None:
    print(f"Annotation: {gene_annotation}")

# == vs is
a = [1, 2, 3]
b = [1, 2, 3]
print(f"\na == b: {a == b}")    # True (same values)
print(f"a is b: {a is b}")     # False (different objects)

---

## 7. Operator Precedence

When multiple operators appear in one expression, Python follows a precedence order (highest to lowest):

| Priority | Operators | Description |
|----------|-----------|-------------|
| 1 (highest) | `**` | Exponentiation |
| 2 | `+x`, `-x`, `~x` | Unary plus, minus, bitwise NOT |
| 3 | `*`, `/`, `//`, `%` | Multiplication, division, modulo |
| 4 | `+`, `-` | Addition, subtraction |
| 5 | `==`, `!=`, `<`, `>`, `<=`, `>=`, `in`, `not in`, `is`, `is not` | Comparisons |
| 6 | `not` | Logical NOT |
| 7 | `and` | Logical AND |
| 8 (lowest) | `or` | Logical OR |

**When in doubt, use parentheses `()` to make your intent clear.**

In [ ]:
# Precedence example with GC content
g = 5
c = 4
total = 20

# These produce different results:
wrong = g + c / total * 100       # division happens first: 5 + (4/20)*100 = 5 + 20 = 25
correct = (g + c) / total * 100   # parentheses force addition first: 9/20*100 = 45

print(f"Without parentheses: {wrong}")   # 25.0 (wrong!)
print(f"With parentheses:    {correct}") # 45.0 (correct!)

In [ ]:
# Logical operator precedence: not > and > or

is_coding = True
gc_ok = True
length_ok = False

# Without parentheses: 'and' binds tighter than 'or'
result1 = is_coding or gc_ok and length_ok
# Evaluated as: is_coding or (gc_ok and length_ok) = True or False = True

# With explicit parentheses for different meaning:
result2 = (is_coding or gc_ok) and length_ok
# Evaluated as: (True or True) and False = True and False = False

print(f"is_coding or gc_ok and length_ok:   {result1}")
print(f"(is_coding or gc_ok) and length_ok: {result2}")

---

## 8. Practical Examples

### Example 1: Melting Temperature of a Short DNA Primer

For short primers (< 14 nt), a simple formula is:  
**Tm = 2(A+T) + 4(G+C)** (degrees Celsius)

For longer primers, the more accurate formula is:  
**Tm = 64.9 + 41 * (G+C - 16.4) / (A+T+G+C)**

In [ ]:
def melting_temperature(primer):
    """Estimate the melting temperature of a DNA primer."""
    primer = primer.upper()
    a = primer.count("A")
    t = primer.count("T")
    g = primer.count("G")
    c = primer.count("C")

    if len(primer) < 14:
        # Wallace rule for short oligos
        tm = 2 * (a + t) + 4 * (g + c)
    else:
        # Basic salt-adjusted formula
        tm = 64.9 + 41 * (g + c - 16.4) / (a + t + g + c)
    return tm

primers = [
    ("Forward", "ATGCGATCGATCGATCGATCG"),
    ("Reverse", "TTACCCGGGAAATTTCCCGGG"),
    ("Short",   "ATGCGATCG"),
]

for name, seq in primers:
    tm = melting_temperature(seq)
    print(f"{name:10s} ({len(seq):2d} nt): Tm = {tm:.1f} C   Seq: {seq}")

### Example 2: Sequence Comparison Dashboard

In [ ]:
seq_a = "ATGAAACCCGGGTTTCCCAAAGGG"
seq_b = "ATGAAAGCCGGGTTTCCCAAAGGG"

print(f"Seq A: {seq_a}")
print(f"Seq B: {seq_b}")
print()

# Length comparison
print(f"Same length?  {len(seq_a) == len(seq_b)}")
print(f"A longer?     {len(seq_a) > len(seq_b)}")

# Content comparison
print(f"Identical?    {seq_a == seq_b}")

# Find mismatches
if len(seq_a) == len(seq_b):
    mismatches = sum(1 for a, b in zip(seq_a, seq_b) if a != b)
    identity = (len(seq_a) - mismatches) / len(seq_a) * 100
    print(f"Mismatches:   {mismatches}")
    print(f"Identity:     {identity:.1f}%")

    # Show alignment with mismatches marked
    print(f"\nAlignment:")
    print(f"  A: {seq_a}")
    match_line = "".join("|" if a == b else "X" for a, b in zip(seq_a, seq_b))
    print(f"     {match_line}")
    print(f"  B: {seq_b}")

### Example 3: Amino Acid Composition Check

In [ ]:
# Check properties of an amino acid sequence
protein = "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPS"

# Hydrophobic amino acids
hydrophobic = set("AILMFWVP")
# Charged amino acids
positive = set("RKH")
negative = set("DE")

hydro_count = sum(1 for aa in protein if aa in hydrophobic)
pos_count = sum(1 for aa in protein if aa in positive)
neg_count = sum(1 for aa in protein if aa in negative)

total = len(protein)
print(f"Protein: {protein}")
print(f"Length: {total} amino acids")
print(f"Hydrophobic: {hydro_count} ({hydro_count/total*100:.1f}%)")
print(f"Positive:    {pos_count} ({pos_count/total*100:.1f}%)")
print(f"Negative:    {neg_count} ({neg_count/total*100:.1f}%)")

# Net charge at neutral pH (rough estimate)
net_charge = pos_count - neg_count
charge_label = "positive" if net_charge > 0 else "negative" if net_charge < 0 else "neutral"
print(f"Net charge:  {net_charge:+d} ({charge_label})")

---

## 9. Exercises

### Exercise 1: GC Content and AT/GC Ratio

For the sequence below, calculate:
1. GC content as a percentage
2. AT/GC ratio
3. Whether the sequence is GC-rich (GC > 50%)

In [ ]:
sequence = "CCTGCGGAAGATCGGCACTAGAATAGCCAGAACCGTTTCTCTGAGGCTTCCGGCCTTCCC"

# Your code here:


### Exercise 2: Stop Codon Finder

Write code that scans the sequence in reading frame 0 (step of 3 from position 0) and prints whether each codon is a start codon, stop codon, or regular codon.

In [ ]:
cds = "ATGAAACCCGGGTAGAAACCC"

# Your code here:
# Expected output (something like):
#   Position 0: ATG -> START
#   Position 3: AAA -> coding
#   Position 6: CCC -> coding
#   Position 9: GGG -> coding
#   Position 12: TAG -> STOP
#   ...


### Exercise 3: Primer Pair Validation

A good PCR primer pair should satisfy these conditions:
- Both primers are 18-25 nucleotides long
- Both have GC content between 40% and 60%
- Their melting temperatures differ by no more than 5 degrees C

Write code that checks whether the primer pair below is valid.

In [ ]:
forward_primer = "ATGCGATCGATCGATCGATCG"
reverse_primer = "TTACCCGGGAAATTTCCCGGG"

# Your code here:
# 1. Check length of both primers
# 2. Calculate GC content of both
# 3. Estimate Tm for both (use the formula from Example 1)
# 4. Print whether the pair passes all criteria


### Exercise 4: Sequence Identity

Given two sequences of equal length, calculate their percent identity (number of matching positions / total length * 100).

In [ ]:
seq1 = "ATGCGATCGATCGATCG"
seq2 = "ATGCAATCGTTCGATCG"

# Your code here:
# 1. Count matching positions
# 2. Calculate percent identity
# 3. Print the result
# Hint: use zip() to iterate over both sequences simultaneously


### Exercise 5: Restriction Enzyme Sites

Find all positions of the EcoRI recognition sequence (`GAATTC`) in the DNA below. Also find BamHI sites (`GGATCC`).

In [ ]:
dna = "ATGCGAATTCGATCGGATCCATCGATCGAATTCATCGATCGGATCCATCG"

# Your code here:
# Find and print all positions of GAATTC (EcoRI) and GGATCC (BamHI)


---

## Summary

| Category | Operators | Key bioinformatics use |
|----------|-----------|------------------------|
| Arithmetic | `+`, `-`, `*`, `/`, `//`, `%`, `**` | GC content, codon math, molecular weight |
| Comparison | `==`, `!=`, `<`, `>`, `<=`, `>=` | Sequence length filtering, threshold checks |
| Logical | `and`, `or`, `not` | Multi-criteria filtering, quality control |
| Membership | `in`, `not in` | Stop codon checks, sequence validation |
| String ops | `+` (concat), `*` (repeat), `[:]` (slice) | Exon joining, repeats, codon extraction |
| Identity | `is`, `is not` | Checking for `None` |

**Key takeaways**:

1. `//` and `%` are your best friends for codon and reading frame calculations
2. Use parentheses to make operator precedence explicit
3. Chained comparisons (`0.4 <= gc <= 0.6`) are more readable than `and`
4. `in` is the Pythonic way to check membership -- use it with sets for speed
5. String slicing `seq[i:i+3]` is the standard idiom for extracting codons

### Next module: Control Flow (if/elif/else, for loops, while loops)

---

[← Previous: Module 2: Variables and Data Types](../../Tier_1_Python_for_Bioinformatics/02_Variables_and_Data_Types/01_variables_and_data_types.ipynb) | [Next: Module 4: Strings and Biological Sequences →](../../Tier_1_Python_for_Bioinformatics/04_Strings_and_Sequences/01_strings_and_sequences.ipynb)